# 5-2 - Topic modeling avec BERTopic (analyse thématique)

In [1]:
# TODO : modifier le chapo si besoin
# TODO : check si les hash tiennent bien.

Lit les embeddings issus de 5_1 et construit un topic model BERTopic à partir
des embeddings pré-calculés et des paramètres fixés (UMAP, HDBSCAN et c-TF-IDF
standards, stopwords français). Aucune réduction automatique du nombre de topics
n'est appliquée, afin de privilégier une lecture manuelle fine.

Explore les topics (mots-clés, hiérarchie, similarités), puis les fusionne sur la base
des similarités et de la lecture des textes pour obtenir 3 niveaux de classification.
Analyse les résultats (topics par affiliation, dynamique temporelle).

Écrit le modèle BERTopic, les figures HTML et l'export final des documents avec topic.

NOTE : les calculs UMAP et BERTopic n'étant pas strictement reproductibles
entre environnements (uv/conda car BLAS/LAPACK diffèrent), l'export complet
du modèle BERTopic est fourni, de même qu'une réduction UMAP pré-calculée.
Le calcul initial a été réalisé dans un environnement conda
(cf. bertopic_repu_env.yml).

In [ ]:
import hashlib
import runpy
import copy
import json
from pathlib import Path
import pandas as pd
import numpy as np
from umap import UMAP  # attention umap-learn pour l'installation
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from stopwordsiso import stopwords
# from bertopic.dimensionality import BaseDimensionalityReduction
# from bertopic.vectorizers import ClassTfidfTransformer

DATASET_DIR = Path(
    "../models/embeddings/"
)  # Chemin de la copie fournie des embeddings pour reproductibilité. Remplacer au besoin si recalcul.

MODEL_DIR = Path("../models/bertopic/qwen3-emb-8B-bertopic-base")

MAPPING_PATH = Path(
    "../models/bertopic/regroupements_topics.py"
)  # path liste des regroupements de topics

# metadonnées des embeddings pour vérifier correspondance avec 5-1
META_PATH = DATASET_DIR / "embeddings_meta.json"

# hash du modèle bertopic pour vérifier correspondance
MODEL_HASH_PATH = Path("../models/bertopic/qwen3-emb-8B-bertopic-base_texts_sha256.txt")

# pré-calcul du UMAP si besoin
# UMAP_PATH = Path("../models/bertopic/umap_nn15_nc5_md0.0_cosine_seed42.npy")

RANDOM_SEED = 42


##############################################################
# Création du modèle BERTopic avec les embeddings pré-calculés
# ou chargement du modèle pré-entraîné si disponible
##############################################################

# ----- Charger docs et precomputed embeddings -----

df = pd.read_csv(DATASET_DIR / "dataset_repu.csv")
docs = df["texte"].tolist()

precomputed_embeddings = np.load(DATASET_DIR / "embeddings.npy")  # Shape : N * 4096

# reduced = np.load(UMAP_PATH)  # si besoin utiliser le UMAP pré-calculé

# ----- Préparer les colonnes utiles pour l'analyse -----
df["dateSeance_ts"] = pd.to_datetime(
    df["dateSeance"].astype(str),
    format="%Y%m%d%H%M%S%f",
)

df["dateSeance_day"] = df["dateSeance_ts"].dt.normalize()

df["affiliation_et_gouv"] = df["affiliation_et_gouv"].fillna("Inconnu")

timestamps = df["dateSeance_ts"].tolist()
affiliations = df["affiliation_et_gouv"].tolist()


# ----- Calculer le hash des textes pour vérifier correspondance -----
def hash_texts(texts) -> str:
    h = hashlib.sha256()
    for t in texts:
        h.update(str(t).encode("utf-8"))
        h.update(b"\x00")  # séparateur, évite que ["ab","c"] == ["a","bc"]
    return h.hexdigest()


current_hash = hash_texts(df["texte"])

# ----- Vérification de la correspondance avec 5-1 -----

if not META_PATH.exists():
    raise FileNotFoundError(f"Métadonnées des embeddings introuvables : {META_PATH}")

with META_PATH.open("r", encoding="utf-8") as f:
    meta = json.load(f)

# Vérification du nombre de textes et d'embeddings
assert len(df) == len(precomputed_embeddings), (
    "Le dataset et les embeddings n'ont pas le même nombre de lignes."
)
# Vérification de la forme des embeddings par rapport au métadonnées
assert list(precomputed_embeddings.shape) == meta["embedding_shape"]

# Vérification que les textes et leur ordre correspondent à 5-1
assert current_hash == meta["texts_sha256"], (
    "Les textes ou leur ordre ne correspondent pas aux embeddings."
)

# ----- Modèle BERTopic de base : chargement ou création -----

if MODEL_DIR.exists():
    print(f"Chargement du modèle BERTopic de base : {MODEL_DIR}")

    topic_model = BERTopic.load(MODEL_DIR)

    assert len(topic_model.topics_) == len(df), (
        "Le modèle BERTopic ne correspond pas au dataset (nombre de lignes). "
        "Supprimer MODEL_DIR pour le recalculer."
    )

    if not MODEL_HASH_PATH.exists():
        raise FileNotFoundError(
            f"{MODEL_HASH_PATH} introuvable : alignement non vérifiable."
        )
    assert MODEL_HASH_PATH.read_text() == current_hash, (
        "Le modèle BERTopic a été entraîné sur d'autres textes. "
        "Supprimer MODEL_DIR pour le recalculer."
    )

else:
    print("Modèle de base absent : création en cours...")
    # ----- Paramètres BERTopic-----

    # UMAP
    # (aussi possible utiliser umap pré-calculé si besoin)

    umap_model = UMAP(
        n_neighbors=15,  # default=15
        n_components=5,  # default=5
        min_dist=0.0,  # default=0.0
        metric="cosine",  # default="cosine"
        random_state=RANDOM_SEED,  # pour reproductibilité
    )

    # HDBSCAN
    # NOTE : version standard car choix fait de ne pas chercher
    # à réduire automatiquement le nombre de topics et privilégier
    # une lecture manuelle fine pour réaliser les choix de fusion.

    # VECTORIZER
    french_stopwords = list(stopwords("fr"))
    vectorizer_model = CountVectorizer(stop_words=french_stopwords)

    # CTFIDF
    # NOTE : version standard car choix fait de privilégier une
    # lecture fine des topics pour déterminer leur nom et contenu.

    # ----- Créer et ajuster le modèle -----

    # initialiser le modèle avec paramètres
    topic_model = BERTopic(
        language="french",  # affecte l'affichage des caractères
        embedding_model=None,  # pas d'embedding_model puisque pré-calculés
        umap_model=umap_model,  # ou si besoin utiliser le pré-calcul umap (reduced)
        vectorizer_model=vectorizer_model,  # enlever les stopwords après embbedings/clustering
    )

    # Fit + transform en utilisant les precomputed_embeddings
    topics, probs = topic_model.fit_transform(
        documents=docs, embeddings=precomputed_embeddings
    )

    # Sauvegarde du modèle de base
    MODEL_DIR.parent.mkdir(parents=True, exist_ok=True)

    topic_model.save(
        MODEL_DIR,
        serialization="safetensors",
        save_ctfidf=True,
        save_embedding_model=False,
    )

    MODEL_HASH_PATH.write_text(current_hash)

    print(f"Modèle sauvegardé : {MODEL_DIR}")

W1126 18:48:32.535000 82259 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
2026-11-26 18:48:34,255 - BERTopic - WARNING: You are loading a BERTopic model without explicitly defining an embedding model. If you want to also load in an embedding model, make sure to use `BERTopic.load(my_model, embedding_model=my_embedding_model)`.


Chargement du modèle BERTopic de base : ../models/bertopic/qwen3-emb-8B-bertopic-base


In [3]:
# # Les principales fonctions à tester pour avoir un aperçu simple :

# topic_model.get_topic_info()
# topic_model.visualize_barchart()
# topic_model.visualize_topics()
# topic_model.visualize_hierarchy()
# topic_model.visualize_documents(df["Texte_clean"].to_list())

# Si modification du libellé et qu'on le veut passer l'argument :
# custom_labels=True

In [4]:
###################################################
# Fusion des topics
###################################################

# ----- Regroupements : chargement + contrôles -----

REGROUPEMENTS = runpy.run_path(MAPPING_PATH)[
    "REGROUPEMENTS"
]  # lecture du dictionnaire (sans les commentaires)

info = topic_model.get_topic_info().set_index("Topic")

# Chaque topic du modèle (hors -1) doit apparaître exactement une fois à chaque niveau.
ids_modele = set(info.index) - {-1}
for niveau, groupes in REGROUPEMENTS.items():
    ids = [topic_id for topic_ids in groupes.values() for topic_id in topic_ids]
    assert len(ids) == len(set(ids)), (
        f"{niveau} : un topic apparaît dans plusieurs groupes"
    )
    assert set(ids) == ids_modele, (
        f"{niveau} : écart entre le modèle et les regroupements. "
        f"Absents des regroupements : {sorted(ids_modele - set(ids))}. "
        f"Absents du modèle : {sorted(set(ids) - ids_modele)}"
    )

# Correspondance topic de base -> libellé pour chacun des trois niveaux.
LIBELLES_TOPICS = {
    niveau: {
        int(topic_id): libelle
        for libelle, topic_ids in groupes.items()
        for topic_id in topic_ids
    }
    for niveau, groupes in REGROUPEMENTS.items()
}
# NOTE
# Injecter un libellé aux outliers (-1) : ne modifie pas le comportement
# des topic_model mais les labels renvoyés pour ajout info dans les datasets
# (et pas avoir des NaN silencieux). N'affecte pas les customs names des TM :
# est exclu plus bas de la corr pour ne pas affecter les topic_model
for mapping in LIBELLES_TOPICS.values():
    mapping[-1] = (
        "-1_Outliers"  # NOTE : choix - Si on ne fixe rien ici renverra des NaN dans df
    )

# Topics du modèle de base, qu'il soit chargé ou nouvellement entraîné.
# (ie topic_model.topics_ vs topics)
df["topic_id_base"] = topic_model.topics_

# Pour les topics 1, 2 et 3, chaque niveau reçoit le libellé du groupe
# attribué manuellement.
# Le topic -1 renvoie l'info fixée au dessus dans mapping[-1] = XXXXXX
for n in (1, 2, 3):
    niveau = f"niveau{n}"
    df[niveau] = df["topic_id_base"].map(LIBELLES_TOPICS[niveau])

# Table indexée par topic de base, utilisée pour l'export.
ref = pd.DataFrame(
    {f"n{n}": pd.Series(LIBELLES_TOPICS[f"niveau{n}"]) for n in (1, 2, 3)}
)
ref.index.name = "Topic"

ref.to_csv(
    "../models/bertopic/ref_topics.csv", index=True
)  # True pour trace de l'index topic

In [5]:
# ----- Fusion simple au niveau N -----


def merge_topics_niveau(n):
    """
    Fusionne les topics au niveau N directement,
    en utilisant les regroupements définis dans REGROUPEMENTS.
    """
    m = copy.deepcopy(topic_model)  # copier le TM pour ne pas modifier l'original
    # Regroupements manuels du niveau demandé
    niveau = f"niveau{n}"  # intercepter le niveau demandé
    groupes = REGROUPEMENTS[niveau]  # identifier les groupes du niveau demandé

    # Extraire uniquement les groupes avec plus d'un topic
    # (pas besoin de fusionner si seul topic du groupe)
    topics_to_merge = [
        topic_ids for topic_ids in groupes.values() if len(topic_ids) > 1
    ]

    # Fusion des topics
    m.merge_topics(docs, topics_to_merge)

    # Après fusion, les IDs ont pu changer.
    # On récupère donc les nouveaux IDs depuis m.topics_.
    corr = pd.DataFrame(
        {
            "id": m.topics_,
            "lib": df[f"niveau{n}"].values,
        }
    ).drop_duplicates()

    # Vérifie qu'un topic fusionné correspond à un seul label manuel
    # `-1`: Exclure les outliers de la correspondance des labels
    # et donc ne pas y toucher dans le bertopic topic_model_nX
    corr = corr[corr["id"] != -1]
    assert corr["id"].is_unique, (
        f"Un topic fusionné correspond à plusieurs labels au niveau {n}"
    )

    # Remplace les noms automatiques par les labels manuels
    m.set_topic_labels(dict(zip(corr["id"], corr["lib"])))

    # Compteurs
    merged_count = len(topics_to_merge)
    unchanged_count = len([g for g in groupes.values() if len(g) == 1])
    print(f"Niveau {n} : {merged_count} fusions, {unchanged_count} inchangés")

    return m


# Appliquer
topic_model_n3 = merge_topics_niveau(3)
topic_model_n2 = merge_topics_niveau(2)
topic_model_n1 = merge_topics_niveau(1)

Niveau 3 : 49 fusions, 5 inchangés
Niveau 2 : 26 fusions, 1 inchangés
Niveau 1 : 10 fusions, 1 inchangés


# Tests

In [6]:
df

,uid,SeanceRef,SessionRef,dateSeance,dateSeanceJour,numSeanceJour,numSeance,typeAssemblee,legislature,session,...,libelleAbrev_commission,libelleAbrege_commission,nombre_mentions_repu,repu_match_valide,emb_id,dateSeance_day,topic_id_base,niveau1,niveau2,niveau3
0,CRSANR5L15S2017E1N001,NaN,NaN,20170704150000000,mardi 04 juillet 2017,Unique,1,AN,15,Première session extraordinaire 2017,...,NaN,NaN,10,True,0,2017-07-04,98,Démocratie parlementaire,Censure/débats parlementaires,Gilets jaunes
1,CRSANR5L15S2017E1N001,NaN,NaN,20170704150000000,mardi 04 juillet 2017,Unique,1,AN,15,Première session extraordinaire 2017,...,CION_DEF,Défense,2,True,1,2017-07-04,98,Démocratie parlementaire,Censure/débats parlementaires,Gilets jaunes
2,CRSANR5L15S2017E1N001,NaN,NaN,20170704150000000,mardi 04 juillet 2017,Unique,1,AN,15,Première session extraordinaire 2017,...,CION_DEF,Défense,8,True,2,2017-07-04,-1,-1_Outliers,-1_Outliers,-1_Outliers
3,CRSANR5L15S2017E1N001,NaN,NaN,20170704150000000,mardi 04 juillet 2017,Unique,1,AN,15,Première session extraordinaire 2017,...,CION_DEF,Défense,1,True,3,2017-07-04,98,Démocratie parlementaire,Censure/débats parlementaires,Gilets jaunes
4,CRSANR5L15S2017E1N001,NaN,NaN,20170704150000000,mardi 04 juillet 2017,Unique,1,AN,15,Première session extraordinaire 2017,...,CION_DEF,Défense,1,True,4,2017-07-04,-1,-1_Outliers,-1_Outliers,-1_Outliers
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10826,CRSANR5L16S2024O1N230,RUANR5L16S2024IDS28423,SCR5A2024O1,20240606090000000,jeudi 06 juin 2024,1,230,AN,16,Session ordinaire 2023-2024,...,CION-ECO,Affaires économiques,1,True,10826,2024-06-06,92,Budget/État-providence,Service public,Hôpital public
10827,CRSANR5L16S2024O1N231,RUANR5L16S2024IDS28424,SCR5A2024O1,20240606150000000,jeudi 06 juin 2024,2,231,AN,16,Session ordinaire 2023-2024,...,CION-SOC,Affaires sociales,2,True,10827,2024-06-06,92,Budget/État-providence,Service public,Hôpital public
10828,CRSANR5L16S2024O1N231,RUANR5L16S2024IDS28424,SCR5A2024O1,20240606150000000,jeudi 06 juin 2024,2,231,AN,16,Session ordinaire 2023-2024,...,CION-CEDU,Affaires culturelles et éducation,2,True,10828,2024-06-06,92,Budget/État-providence,Service public,Hôpital public
10829,CRSANR5L16S2024O1N233,RUANR5L16S2024IDS28426,SCR5A2024O1,20240607100000000,vendredi 07 juin 2024,1,233,AN,16,Session ordinaire 2023-2024,...,CION-ECO,Affaires économiques,1,True,10829,2024-06-07,92,Budget/État-providence,Service public,Hôpital public


In [7]:
# # Les principales fonctions à tester pour avoir un aperçu simple :

# topic_model.get_topic_info()
# topic_model.visualize_barchart()
# topic_model.visualize_topics()
# topic_model.visualize_hierarchy()
# topic_model.visualize_documents(df["Texte_clean"].to_list())

# Si modification du libellé et qu'on le veut passer l'argument :
# custom_labels=True

## Topics

In [8]:
topic_model_n3.get_topic_info()

,Topic,Count,Name,CustomName,Representation,Representative_Docs
0,-1,2279,-1_république_ministre_loi_france,-1_république_ministre_loi_france,"[république, ministre, loi, france, monsieur, ...",[Le groupe UDI-Agir et indépendants ne votera ...
1,0,448,0_président_république_parlement_parlementaire,Fonctionnement/réglement de l’assemblée,"[président, république, parlement, parlementai...",[S'il tient davantage de Pompidou que de Lecan...
2,1,414,1_élus_collectivités_territoires_maires,Collectivités et élus locaux,"[élus, collectivités, territoires, maires, com...",[La crise démocratique que traverse notre pays...
3,2,406,2_49_retraite_censure_sociale,Blocage parlementaire budget/retraites,"[49, retraite, censure, sociale, gouvernement,...","[Mercredi dernier, j'ai décidé d'engager la re..."
4,3,347,3_république_républicain_ve_arc,Mots-slogans,"[république, républicain, ve, arc, amendement,...","[Même ceux qui, pour vous, sont en dehors de l..."
5,4,340,4_associations_engagement_association_contrat,Associations,"[associations, engagement, association, contra...","[Monsieur le président, monsieur le ministre, ..."
6,5,280,5_école_instruction_enfants_famille,Instruction/autorité,"[école, instruction, enfants, famille, harcèle...","[Madame la présidente, monsieur le ministre, m..."
7,6,263,6_fiscale_impôt_milliards_fraude,Fiscalité,"[fiscale, impôt, milliards, fraude, finances, ...","[Monsieur le président, monsieur le ministre, ..."
8,7,262,7_école_élèves_éducation_enseignants,Éducation,"[école, élèves, éducation, enseignants, scolai...","[Parfois, la République a rendez-vous avec ell..."
9,8,247,8_femmes_violences_égalité_loi,Discriminations de genre,"[femmes, violences, égalité, loi, hommes, vict...","[Tous les trois jours, une femme meurt sous le..."


## Barchart

In [9]:
# topic_model_n3.visualize_barchart(
#     # n_words=10,  # Select the number of words to display per topic
#     # topics = [0,1,2,3,4], # Select specific topics to display
#     top_n_topics=8,  # Select the first n topics to display
#     # height = 300, # Adjust the height of the plot
#     # width = 800 # Adjust the width of the plot
# )

## Visualisation documents / topics

In [10]:
# # TODO : si utilisé pour finaliser une carte, figer une projection umap 2D pour reproductibilité ?
# topic_model.visualize_documents(
#     docs=docs,
#     embeddings=precomputed_embeddings,
#     hide_annotations=True,  # better readability
#     # topics=[0, 1, 2, 3],  # Select topics to highlight
#     # height = 300, # Adjust the height of the plot
#     # width = 800 # Adjust the width of the plot
# )

## Hiérarchie des topics

In [11]:
# topic_model.visualize_hierarchy()

In [12]:
# hierarchical_topics = topic_model.hierarchical_topics(docs)
# print(topic_model.get_topic_tree(hierarchical_topics))

In [13]:
# topic_model.visualize_heatmap()

# Vrac

Comparaison de modèles :

https://maartengr.github.io/BERTopic/getting_started/tips_and_tricks/tips_and_tricks.html#finding-similar-topics-between-models

## Dynamic topic model

* `global_tuning`
  * Whether to average the topic representation of a topic at time *t* with its global topic representation
* `evolution_tuning`
  * Whether to average the topic representation of a topic at time *t* with the topic representation of that topic at time *t-1*
* `nr_bins`
  * The number of bins to put our timestamps into. It is computationally inefficient to extract the topics at thousands of different timestamps. Therefore, it is advised to keep this value below 20.


In [14]:
# # NOTE : timestamps = df["dateSeance_day"].tolist()
# # Analyse temporelle
# topics_over_time = topic_model_n1.topics_over_time(
#     docs=docs,
#     timestamps=timestamps,
#     global_tuning=True,
#     evolution_tuning=True,
#     nr_bins=20,
# )

# fig_dynamic_topic = topic_model_n1.visualize_topics_over_time(
#     topics_over_time,
#     top_n_topics=10,
# )

# fig_dynamic_topic

In [15]:
# fig_dynamic_topic.write_html("../reports/figures/dynamic_topics.html")

### Topics per class

In [16]:
# # ATTENTION, PLANTAIT À CAUSE DES NA
# # cf ajout précédent d'une classe "Inconnu" pour les NA dans affiliation_et_gouv
# topics_per_class = topic_model.topics_per_class(docs, classes=affiliations)

# fig_topics_per_class = topic_model.visualize_topics_per_class(
#     topics_per_class,
#     # topics=[0, 1, 2, 3],  # choix spécifique de topics à visualiser
#     top_n_topics=11,  # choix des X principaux topics
#     custom_labels=True,
# )
# fig_topics_per_class

In [17]:
# fig_topics_per_class.write_html("../reports/figures/topics_per_class.html")

In [18]:
# TODO: revoir regroupement des topics
# TODO: revoir Topic distribution
# TODO: aviser alternatives representation sur le nom des topics ?

In [19]:
# df["topic"] = topics
# df["probability"] = probs

# # joindre le nom/label du topic plutôt que juste l'ID
# topic_info = topic_model.get_topic_info()[["Topic", "Name"]]
# df = df.merge(topic_info, left_on="topic", right_on="Topic", how="left")

# # df.to_csv("../models/bertopic/export_final.csv", index=False)

In [20]:
df

,uid,SeanceRef,SessionRef,dateSeance,dateSeanceJour,numSeanceJour,numSeance,typeAssemblee,legislature,session,...,libelleAbrev_commission,libelleAbrege_commission,nombre_mentions_repu,repu_match_valide,emb_id,dateSeance_day,topic_id_base,niveau1,niveau2,niveau3
0,CRSANR5L15S2017E1N001,NaN,NaN,20170704150000000,mardi 04 juillet 2017,Unique,1,AN,15,Première session extraordinaire 2017,...,NaN,NaN,10,True,0,2017-07-04,98,Démocratie parlementaire,Censure/débats parlementaires,Gilets jaunes
1,CRSANR5L15S2017E1N001,NaN,NaN,20170704150000000,mardi 04 juillet 2017,Unique,1,AN,15,Première session extraordinaire 2017,...,CION_DEF,Défense,2,True,1,2017-07-04,98,Démocratie parlementaire,Censure/débats parlementaires,Gilets jaunes
2,CRSANR5L15S2017E1N001,NaN,NaN,20170704150000000,mardi 04 juillet 2017,Unique,1,AN,15,Première session extraordinaire 2017,...,CION_DEF,Défense,8,True,2,2017-07-04,-1,-1_Outliers,-1_Outliers,-1_Outliers
3,CRSANR5L15S2017E1N001,NaN,NaN,20170704150000000,mardi 04 juillet 2017,Unique,1,AN,15,Première session extraordinaire 2017,...,CION_DEF,Défense,1,True,3,2017-07-04,98,Démocratie parlementaire,Censure/débats parlementaires,Gilets jaunes
4,CRSANR5L15S2017E1N001,NaN,NaN,20170704150000000,mardi 04 juillet 2017,Unique,1,AN,15,Première session extraordinaire 2017,...,CION_DEF,Défense,1,True,4,2017-07-04,-1,-1_Outliers,-1_Outliers,-1_Outliers
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10826,CRSANR5L16S2024O1N230,RUANR5L16S2024IDS28423,SCR5A2024O1,20240606090000000,jeudi 06 juin 2024,1,230,AN,16,Session ordinaire 2023-2024,...,CION-ECO,Affaires économiques,1,True,10826,2024-06-06,92,Budget/État-providence,Service public,Hôpital public
10827,CRSANR5L16S2024O1N231,RUANR5L16S2024IDS28424,SCR5A2024O1,20240606150000000,jeudi 06 juin 2024,2,231,AN,16,Session ordinaire 2023-2024,...,CION-SOC,Affaires sociales,2,True,10827,2024-06-06,92,Budget/État-providence,Service public,Hôpital public
10828,CRSANR5L16S2024O1N231,RUANR5L16S2024IDS28424,SCR5A2024O1,20240606150000000,jeudi 06 juin 2024,2,231,AN,16,Session ordinaire 2023-2024,...,CION-CEDU,Affaires culturelles et éducation,2,True,10828,2024-06-06,92,Budget/État-providence,Service public,Hôpital public
10829,CRSANR5L16S2024O1N233,RUANR5L16S2024IDS28426,SCR5A2024O1,20240607100000000,vendredi 07 juin 2024,1,233,AN,16,Session ordinaire 2023-2024,...,CION-ECO,Affaires économiques,1,True,10829,2024-06-07,92,Budget/État-providence,Service public,Hôpital public


## Sauvegarde

In [21]:
# Sauvegarde info topics + sortie df
# TODO : path et names à affiner
dossier = Path("../models/bertopic/topic_info")
dossier.mkdir(parents=True, exist_ok=True)

modeles = {
    "base": topic_model,
    "niveau1": topic_model_n1,
    "niveau2": topic_model_n2,
    "niveau3": topic_model_n3,
}

for nom, modele in modeles.items():
    # écrire csv info
    modele.get_topic_info().to_csv(dossier / f"topics_{nom}.csv", index=False)
    # ajouter id par niveau dans df (sauf base qu'on a déjà)
    if nom != "base":
        df[f"topic_id_{nom}"] = modele.topics_

df.to_csv("../models/bertopic/documents_topics.csv", index=False)  # sortir le full csv

In [22]:
df

,uid,SeanceRef,SessionRef,dateSeance,dateSeanceJour,numSeanceJour,numSeance,typeAssemblee,legislature,session,...,repu_match_valide,emb_id,dateSeance_day,topic_id_base,niveau1,niveau2,niveau3,topic_id_niveau1,topic_id_niveau2,topic_id_niveau3
0,CRSANR5L15S2017E1N001,NaN,NaN,20170704150000000,mardi 04 juillet 2017,Unique,1,AN,15,Première session extraordinaire 2017,...,True,0,2017-07-04,98,Démocratie parlementaire,Censure/débats parlementaires,Gilets jaunes,0,0,44
1,CRSANR5L15S2017E1N001,NaN,NaN,20170704150000000,mardi 04 juillet 2017,Unique,1,AN,15,Première session extraordinaire 2017,...,True,1,2017-07-04,98,Démocratie parlementaire,Censure/débats parlementaires,Gilets jaunes,0,0,44
2,CRSANR5L15S2017E1N001,NaN,NaN,20170704150000000,mardi 04 juillet 2017,Unique,1,AN,15,Première session extraordinaire 2017,...,True,2,2017-07-04,-1,-1_Outliers,-1_Outliers,-1_Outliers,-1,-1,-1
3,CRSANR5L15S2017E1N001,NaN,NaN,20170704150000000,mardi 04 juillet 2017,Unique,1,AN,15,Première session extraordinaire 2017,...,True,3,2017-07-04,98,Démocratie parlementaire,Censure/débats parlementaires,Gilets jaunes,0,0,44
4,CRSANR5L15S2017E1N001,NaN,NaN,20170704150000000,mardi 04 juillet 2017,Unique,1,AN,15,Première session extraordinaire 2017,...,True,4,2017-07-04,-1,-1_Outliers,-1_Outliers,-1_Outliers,-1,-1,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10826,CRSANR5L16S2024O1N230,RUANR5L16S2024IDS28423,SCR5A2024O1,20240606090000000,jeudi 06 juin 2024,1,230,AN,16,Session ordinaire 2023-2024,...,True,10826,2024-06-06,92,Budget/État-providence,Service public,Hôpital public,1,4,25
10827,CRSANR5L16S2024O1N231,RUANR5L16S2024IDS28424,SCR5A2024O1,20240606150000000,jeudi 06 juin 2024,2,231,AN,16,Session ordinaire 2023-2024,...,True,10827,2024-06-06,92,Budget/État-providence,Service public,Hôpital public,1,4,25
10828,CRSANR5L16S2024O1N231,RUANR5L16S2024IDS28424,SCR5A2024O1,20240606150000000,jeudi 06 juin 2024,2,231,AN,16,Session ordinaire 2023-2024,...,True,10828,2024-06-06,92,Budget/État-providence,Service public,Hôpital public,1,4,25
10829,CRSANR5L16S2024O1N233,RUANR5L16S2024IDS28426,SCR5A2024O1,20240607100000000,vendredi 07 juin 2024,1,233,AN,16,Session ordinaire 2023-2024,...,True,10829,2024-06-07,92,Budget/État-providence,Service public,Hôpital public,1,4,25
